#Парсинг публикаций новостных сообществ социальной сети ВКонтакте

##Написание функции по извлечению данных с помощью VK API

In [ ]:
pip install vk_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.1 MB/s eta 0:00:00


In [ ]:
api_token = #'Вставьте свой ключ доступа'
# используемый личный ключ скрыт из ячейки в целях безопасности аккаунта в соцсети

In [ ]:
# импорт библиотек
import vk_api
from datetime import datetime
import time
import pandas as pd

# инициализация функции для извлечения публикаций
def get_posts(api_token, group_id, count=100, start_offset=5000):
  '''
  Получает тексты постов и пользовательские реакции из сообществ с сайта https://vk.com.
  Собирает посты, пока не дойдет до даты 01.01.2020, либо пока не закончатся посты.

  Параметры:
    api_token: ключ доступа для выполнения запросов к VK API
    group_id: уникальный идентификатор сообщества, начинающийся с '-'
    count: число выгружаемых постов за 1 итерацию
    start_offset: начальная позиция со смещением (сколько последних постов пропускаем перед сбором данных)
  Вывод функции:
    Возвращает all_posts – список из словарей; в каждом словаре содержится информация
                           о конкретной публикации.
  '''
  # инициализация VK API
  vk_session = vk_api.VkApi(token=api_token)
  vk = vk_session.get_api()

  # инициализация словаря, смещения, целевой даты и флага ее достижения
  all_posts = []
  offset = start_offset
  target_date = datetime(2020, 1, 1).timestamp()
  target_flag = False

  print(f'Начинаем сбор с offset = {offset}')
  print(f'Цель: дойти до 01.01.2020')
  print('-' * 50)

  # пока не достигнута целевая дата, цикл в работе
  while not target_flag:
    # отправка запроса к новостной 'стене' сообщества
    response = vk.wall.get(owner_id=group_id,
                           filter='owner',
                           count=count,
                           offset=offset)
    # пустой ответ на запрос сигнализирует об отсутствии постов
    if not response.get('items'):
      print('Постов больше нет')
      break
    # извлечение даты поста из ответа на запрос
    for post in response['items']:
      post_date = post['date']
      post_date_string = datetime.fromtimestamp(post_date).strftime('%Y-%m-%d')
      post_year = datetime.fromtimestamp(post_date).year
      # проверка на достижении целевой даты 01.01.2020
      if post_date <= target_date:
        target_flag = True
        if post_year == 2020 or post_year < 2020:
          print(f'\n- Достигли {post_year} года -')
          print(f'Последний пост: {post['id']} от {post_date_string}')
          break
      # извлечение данных из ответа на запрос в виде словаря и присоединение их к списку со всеми постами
      else:
        all_posts.append({'post_id': post['id'],
                          'date': post_date_string,
                          'text': post['text'],
                          'likes': post.get('likes', {}).get('count', 0),
                          'comments_count': post.get('comments', {}).get('count', 0),
                          'reposts': post.get('reposts', {}).get('count', 0),
                          'views': post.get('views', {}).get('count', 0) if 'views' in post else 0})
    # завершение цикла в случае достижения целевой даты 01.01.2020
    if target_flag:
      break

    print(f'Обработано offset {offset}, собрано {len(all_posts)} постов (последний: {post_date_string})')
    # если цикл продолжается, значение смещения обновляется на число собранных постов (100)
    offset += 100
    # задержка алгоритма для избежания превышения установленных лимитов запросов (rps)
    time.sleep(0.1)

  return all_posts

##Применение функции для выгрузки данных из новостных сообществ

In [ ]:
# https://vk.com/vedomosti
group_id = '-15548215'
name = 'vedomosti.csv'

posts = get_posts(api_token, group_id, start_offset=5000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=5000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 5000, собрано 100 постов (последний: 2026-01-03)
Обработано offset 5100, собрано 200 постов (последний: 2025-12-31)
Обработано offset 5200, собрано 300 постов (последний: 2025-12-28)
Обработано offset 5300, собрано 400 постов (последний: 2025-12-26)
Обработано offset 5400, собрано 500 постов (последний: 2025-12-23)
Обработано offset 5500, собрано 600 постов (последний: 2025-12-21)
Обработано offset 5600, собрано 700 постов (последний: 2025-12-18)
Обработано offset 5700, собрано 800 постов (последний: 2025-12-15)
Обработано offset 5800, собрано 900 постов (последний: 2025-12-12)
Обработано offset 5900, собрано 1000 постов (последний: 2025-12-09)
Обработано offset 6000, собрано 1100 постов (последний: 2025-12-06)
Обработано offset 6100, собрано 1200 постов (последний: 2025-12-03)
Обработано offset 6200, собрано 1300 постов (последний: 2025-11-30)
Обработано offset 63

In [ ]:
# https://vk.com/wow.mosnow
group_id = '-207215539'
name = 'mosnow.csv'

posts = get_posts(api_token, group_id, start_offset=200)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=200
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 200, собрано 100 постов (последний: 2025-11-01)
Обработано offset 300, собрано 200 постов (последний: 2025-09-09)
Обработано offset 400, собрано 300 постов (последний: 2025-07-20)
Обработано offset 500, собрано 400 постов (последний: 2025-05-31)
Обработано offset 600, собрано 500 постов (последний: 2025-04-10)
Обработано offset 700, собрано 600 постов (последний: 2025-02-19)
Обработано offset 800, собрано 700 постов (последний: 2025-01-01)
Обработано offset 900, собрано 800 постов (последний: 2024-05-12)
Обработано offset 1000, собрано 900 постов (последний: 2024-04-03)
Обработано offset 1100, собрано 1000 постов (последний: 2024-03-13)
Обработано offset 1200, собрано 1100 постов (последний: 2024-02-28)
Обработано offset 1300, собрано 1200 постов (последний: 2024-02-08)
Обработано offset 1400, собрано 1300 постов (последний: 2024-01-23)
Обработано offset 1500, собра

In [ ]:
# https://vk.com/m24
group_id = '-35068738'
name = 'm24.csv'

posts = get_posts(api_token, group_id, start_offset=4000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=4000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 4000, собрано 100 постов (последний: 2026-01-01)
Обработано offset 4100, собрано 200 постов (последний: 2025-12-28)
Обработано offset 4200, собрано 300 постов (последний: 2025-12-24)
Обработано offset 4300, собрано 400 постов (последний: 2025-12-20)
Обработано offset 4400, собрано 500 постов (последний: 2025-12-18)
Обработано offset 4500, собрано 600 постов (последний: 2025-12-14)
Обработано offset 4600, собрано 700 постов (последний: 2025-12-10)
Обработано offset 4700, собрано 800 постов (последний: 2025-12-06)
Обработано offset 4800, собрано 900 постов (последний: 2025-12-03)
Обработано offset 4900, собрано 1000 постов (последний: 2025-11-29)
Обработано offset 5000, собрано 1100 постов (последний: 2025-11-26)
Обработано offset 5100, собрано 1200 постов (последний: 2025-11-22)
Обработано offset 5200, собрано 1300 постов (последний: 2025-11-18)
Обработано offset 53

In [ ]:
# https://vk.com/onmoscow
group_id = '-204915679'
name = 'onmoscow.csv'

posts = get_posts(api_token, group_id, start_offset=1000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1000, собрано 100 постов (последний: 2026-01-24)
Обработано offset 1100, собрано 200 постов (последний: 2026-01-16)
Обработано offset 1200, собрано 300 постов (последний: 2026-01-08)
Обработано offset 1300, собрано 400 постов (последний: 2025-12-31)
Обработано offset 1400, собрано 500 постов (последний: 2025-12-24)
Обработано offset 1500, собрано 600 постов (последний: 2025-12-17)
Обработано offset 1600, собрано 700 постов (последний: 2025-12-10)
Обработано offset 1700, собрано 800 постов (последний: 2025-12-02)
Обработано offset 1800, собрано 900 постов (последний: 2025-11-24)
Обработано offset 1900, собрано 1000 постов (последний: 2025-11-16)
Обработано offset 2000, собрано 1100 постов (последний: 2025-11-08)
Обработано offset 2100, собрано 1200 постов (последний: 2025-10-31)
Обработано offset 2200, собрано 1300 постов (последний: 2025-10-23)
Обработано offset 23

In [ ]:
# https://vk.com/moscowtop1
group_id = '-126212725'
name = 'moscowtop1.csv'

posts = get_posts(api_token, group_id, start_offset=1500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1500, собрано 100 постов (последний: 2026-01-02)
Обработано offset 1600, собрано 200 постов (последний: 2025-12-26)
Обработано offset 1700, собрано 300 постов (последний: 2025-12-19)
Обработано offset 1800, собрано 400 постов (последний: 2025-12-12)
Обработано offset 1900, собрано 500 постов (последний: 2025-12-05)
Обработано offset 2000, собрано 600 постов (последний: 2025-11-28)
Обработано offset 2100, собрано 700 постов (последний: 2025-11-21)
Обработано offset 2200, собрано 800 постов (последний: 2025-11-14)
Обработано offset 2300, собрано 900 постов (последний: 2025-11-07)
Обработано offset 2400, собрано 1000 постов (последний: 2025-10-31)
Обработано offset 2500, собрано 1100 постов (последний: 2025-10-24)
Обработано offset 2600, собрано 1200 постов (последний: 2025-10-17)
Обработано offset 2700, собрано 1300 постов (последний: 2025-10-10)
Обработано offset 28

In [ ]:
# https://vk.com/moscowmapvideo
group_id = '-33234296'
name = 'moscowmapvideo.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-11-17)
Обработано offset 100, собрано 200 постов (последний: 2025-09-23)
Обработано offset 200, собрано 300 постов (последний: 2025-06-29)
Обработано offset 300, собрано 400 постов (последний: 2025-06-20)
Обработано offset 400, собрано 500 постов (последний: 2025-06-09)
Обработано offset 500, собрано 600 постов (последний: 2025-05-30)
Обработано offset 600, собрано 700 постов (последний: 2025-05-20)
Обработано offset 700, собрано 800 постов (последний: 2025-05-11)
Обработано offset 800, собрано 900 постов (последний: 2025-05-02)
Обработано offset 900, собрано 1000 постов (последний: 2025-04-22)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-12)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-31)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-20)
Обработано offset 1300, собрано 140

In [ ]:
# https://vk.com/novosti_moscow_vk
group_id = '-166856580'
name = 'novosti_moscow_vk.csv'

posts = get_posts(api_token, group_id, start_offset=1500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1500, собрано 100 постов (последний: 2026-01-14)
Обработано offset 1600, собрано 200 постов (последний: 2026-01-08)
Обработано offset 1700, собрано 300 постов (последний: 2026-01-02)
Обработано offset 1800, собрано 400 постов (последний: 2025-12-27)
Обработано offset 1900, собрано 500 постов (последний: 2025-12-21)
Обработано offset 2000, собрано 600 постов (последний: 2025-12-15)
Обработано offset 2100, собрано 700 постов (последний: 2025-12-09)
Обработано offset 2200, собрано 800 постов (последний: 2025-12-04)
Обработано offset 2300, собрано 900 постов (последний: 2025-11-28)
Обработано offset 2400, собрано 1000 постов (последний: 2025-11-22)
Обработано offset 2500, собрано 1100 постов (последний: 2025-11-16)
Обработано offset 2600, собрано 1200 постов (последний: 2025-11-11)
Обработано offset 2700, собрано 1300 постов (последний: 2025-11-05)
Обработано offset 28

In [ ]:
# https://vk.com/tvcmoscow
group_id = '-195924008'
name = 'tvcmoscow.csv'

posts = get_posts(api_token, group_id, start_offset=1500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')


Начинаем сбор с offset=1500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1500, собрано 100 постов (последний: 2026-01-24)
Обработано offset 1600, собрано 200 постов (последний: 2026-01-19)
Обработано offset 1700, собрано 300 постов (последний: 2026-01-14)
Обработано offset 1800, собрано 400 постов (последний: 2026-01-08)
Обработано offset 1900, собрано 500 постов (последний: 2026-01-02)
Обработано offset 2000, собрано 600 постов (последний: 2025-12-25)
Обработано offset 2100, собрано 700 постов (последний: 2025-12-21)
Обработано offset 2200, собрано 800 постов (последний: 2025-12-14)
Обработано offset 2300, собрано 900 постов (последний: 2025-12-08)
Обработано offset 2400, собрано 1000 постов (последний: 2025-12-01)
Обработано offset 2500, собрано 1100 постов (последний: 2025-11-25)
Обработано offset 2600, собрано 1200 постов (последний: 2025-11-19)
Обработано offset 2700, собрано 1300 постов (последний: 2025-11-12)
Обработано offset 2

In [ ]:
# https://vk.com/moskvwa
group_id = '-16348810'
name = 'moskvwa.csv'

posts = get_posts(api_token, group_id, start_offset=500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 500, собрано 100 постов (последний: 2026-02-04)
Обработано offset 600, собрано 200 постов (последний: 2026-01-25)
Обработано offset 700, собрано 300 постов (последний: 2026-01-15)
Обработано offset 800, собрано 400 постов (последний: 2026-01-04)
Обработано offset 900, собрано 500 постов (последний: 2025-12-24)
Обработано offset 1000, собрано 600 постов (последний: 2025-12-14)
Обработано offset 1100, собрано 700 постов (последний: 2025-12-04)
Обработано offset 1200, собрано 800 постов (последний: 2025-11-24)
Обработано offset 1300, собрано 900 постов (последний: 2025-11-14)
Обработано offset 1400, собрано 1000 постов (последний: 2025-11-04)
Обработано offset 1500, собрано 1100 постов (последний: 2025-10-25)
Обработано offset 1600, собрано 1200 постов (последний: 2025-10-15)
Обработано offset 1700, собрано 1300 постов (последний: 2025-10-05)
Обработано offset 1800, со

In [ ]:
# https://vk.com/mnews_ru
group_id = '-193079545'
name = 'mnews_ru.csv'

posts = get_posts(api_token, group_id, start_offset=500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 500, собрано 100 постов (последний: 2026-01-04)
Обработано offset 600, собрано 200 постов (последний: 2025-12-08)
Обработано offset 700, собрано 300 постов (последний: 2025-11-15)
Обработано offset 800, собрано 400 постов (последний: 2025-10-24)
Обработано offset 900, собрано 500 постов (последний: 2025-09-30)
Обработано offset 1000, собрано 600 постов (последний: 2025-09-04)
Обработано offset 1100, собрано 700 постов (последний: 2025-07-22)
Обработано offset 1200, собрано 800 постов (последний: 2025-06-18)
Обработано offset 1300, собрано 900 постов (последний: 2025-05-20)
Обработано offset 1400, собрано 1000 постов (последний: 2025-04-24)
Обработано offset 1500, собрано 1100 постов (последний: 2025-04-06)
Обработано offset 1600, собрано 1200 постов (последний: 2025-03-19)
Обработано offset 1700, собрано 1300 постов (последний: 2025-03-07)
Обработано offset 1800, со

In [ ]:
# https://vk.com/msk_mine
group_id = '-62438886'
name = 'msk_mine.csv'

posts = get_posts(api_token, group_id, start_offset=500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 500, собрано 100 постов (последний: 2026-02-15)
Обработано offset 600, собрано 200 постов (последний: 2026-02-03)
Обработано offset 700, собрано 300 постов (последний: 2026-01-22)
Обработано offset 800, собрано 400 постов (последний: 2026-01-10)
Обработано offset 900, собрано 500 постов (последний: 2025-12-25)
Обработано offset 1000, собрано 600 постов (последний: 2025-11-29)
Обработано offset 1100, собрано 700 постов (последний: 2025-11-11)
Обработано offset 1200, собрано 800 постов (последний: 2025-10-31)
Обработано offset 1300, собрано 900 постов (последний: 2025-10-20)
Обработано offset 1400, собрано 1000 постов (последний: 2025-10-08)
Обработано offset 1500, собрано 1100 постов (последний: 2025-09-27)
Обработано offset 1600, собрано 1200 постов (последний: 2025-09-17)
Обработано offset 1700, собрано 1300 постов (последний: 2025-09-06)
Обработано offset 1800, со

In [ ]:
# https://vk.com/mosc1
group_id = '-34274053'
name = 'mosc1.csv'

posts = get_posts(api_token, group_id, start_offset=500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=800
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 800, собрано 100 постов (последний: 2026-01-02)
Обработано offset 900, собрано 200 постов (последний: 2025-12-22)
Обработано offset 1000, собрано 300 постов (последний: 2025-12-12)
Обработано offset 1100, собрано 400 постов (последний: 2025-12-01)
Обработано offset 1200, собрано 500 постов (последний: 2025-11-20)
Обработано offset 1300, собрано 600 постов (последний: 2025-11-09)
Обработано offset 1400, собрано 700 постов (последний: 2025-10-29)
Обработано offset 1500, собрано 800 постов (последний: 2025-10-19)
Обработано offset 1600, собрано 900 постов (последний: 2025-10-07)
Обработано offset 1700, собрано 1000 постов (последний: 2025-09-26)
Обработано offset 1800, собрано 1100 постов (последний: 2025-09-15)
Обработано offset 1900, собрано 1200 постов (последний: 2025-09-05)
Обработано offset 2000, собрано 1300 постов (последний: 2025-08-25)
Обработано offset 2100,

In [ ]:
# https://vk.com/msk4you
group_id = '-59570143'
name = 'msk4you.csv'

posts = get_posts(api_token, group_id, start_offset=500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 500, собрано 100 постов (последний: 2026-02-09)
Обработано offset 600, собрано 200 постов (последний: 2026-01-27)
Обработано offset 700, собрано 300 постов (последний: 2026-01-14)
Обработано offset 800, собрано 400 постов (последний: 2025-12-28)
Обработано offset 900, собрано 500 постов (последний: 2025-12-13)
Обработано offset 1000, собрано 600 постов (последний: 2025-11-13)
Обработано offset 1100, собрано 700 постов (последний: 2025-10-31)
Обработано offset 1200, собрано 800 постов (последний: 2025-10-17)
Обработано offset 1300, собрано 900 постов (последний: 2025-10-04)
Обработано offset 1400, собрано 1000 постов (последний: 2025-09-20)
Обработано offset 1500, собрано 1100 постов (последний: 2025-09-07)
Обработано offset 1600, собрано 1200 постов (последний: 2025-08-24)
Обработано offset 1700, собрано 1300 постов (последний: 2025-08-12)
Обработано offset 1800, со

In [ ]:
# https://vk.com/moyamsk
group_id = '-41890491'
name = 'moyamsk.csv'

posts = get_posts(api_token, group_id, start_offset=1000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')


Начинаем сбор с offset=1000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1000, собрано 100 постов (последний: 2026-01-21)
Обработано offset 1100, собрано 200 постов (последний: 2026-01-13)
Обработано offset 1200, собрано 300 постов (последний: 2026-01-05)
Обработано offset 1300, собрано 400 постов (последний: 2025-12-27)
Обработано offset 1400, собрано 500 постов (последний: 2025-12-16)
Обработано offset 1500, собрано 600 постов (последний: 2025-12-08)
Обработано offset 1600, собрано 700 постов (последний: 2025-12-01)
Обработано offset 1700, собрано 800 постов (последний: 2025-11-23)
Обработано offset 1800, собрано 900 постов (последний: 2025-11-16)
Обработано offset 1900, собрано 1000 постов (последний: 2025-11-08)
Обработано offset 2000, собрано 1100 постов (последний: 2025-10-31)
Обработано offset 2100, собрано 1200 постов (последний: 2025-10-22)
Обработано offset 2200, собрано 1300 постов (последний: 2025-10-14)
Обработано offset 2

In [ ]:
# https://vk.com/typical_msk
group_id = '-32112479'
name = 'typical_msk.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-12-23)
Обработано offset 100, собрано 200 постов (последний: 2025-11-13)
Обработано offset 200, собрано 300 постов (последний: 2025-10-07)
Обработано offset 300, собрано 400 постов (последний: 2025-09-03)
Обработано offset 400, собрано 500 постов (последний: 2025-07-29)
Обработано offset 500, собрано 600 постов (последний: 2025-06-18)
Обработано offset 600, собрано 700 постов (последний: 2025-05-10)
Обработано offset 700, собрано 800 постов (последний: 2025-04-07)
Обработано offset 800, собрано 900 постов (последний: 2025-03-03)
Обработано offset 900, собрано 1000 постов (последний: 2025-01-27)
Обработано offset 1000, собрано 1100 постов (последний: 2024-12-27)
Обработано offset 1100, собрано 1200 постов (последний: 2024-11-26)
Обработано offset 1200, собрано 1300 постов (последний: 2024-10-23)
Обработано offset 1300, собрано 140

In [ ]:
# https://vk.com/mskagency
group_id = '-35460441'
name = 'mskagency.csv'

posts = get_posts(api_token, group_id, start_offset=700)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=700
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 700, собрано 100 постов (последний: 2026-02-01)
Обработано offset 800, собрано 200 постов (последний: 2026-01-19)
Обработано offset 900, собрано 300 постов (последний: 2025-12-30)
Обработано offset 1000, собрано 400 постов (последний: 2025-12-18)
Обработано offset 1100, собрано 500 постов (последний: 2025-12-04)
Обработано offset 1200, собрано 600 постов (последний: 2025-11-18)
Обработано offset 1300, собрано 700 постов (последний: 2025-11-01)
Обработано offset 1400, собрано 800 постов (последний: 2025-10-15)
Обработано offset 1500, собрано 900 постов (последний: 2025-09-27)
Обработано offset 1600, собрано 1000 постов (последний: 2025-09-11)
Обработано offset 1700, собрано 1100 постов (последний: 2025-08-27)
Обработано offset 1800, собрано 1200 постов (последний: 2025-08-07)
Обработано offset 1900, собрано 1300 постов (последний: 2025-07-16)
Обработано offset 2000, 

In [ ]:
# https://vk.com/vmdaily
group_id = '-29487103'
name = 'vmdaily.csv'

posts = get_posts(api_token, group_id, start_offset=1000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1000, собрано 100 постов (последний: 2026-01-02)
Обработано offset 1100, собрано 200 постов (последний: 2025-12-23)
Обработано offset 1200, собрано 300 постов (последний: 2025-12-15)
Обработано offset 1300, собрано 400 постов (последний: 2025-12-05)
Обработано offset 1400, собрано 500 постов (последний: 2025-11-24)
Обработано offset 1500, собрано 600 постов (последний: 2025-11-14)
Обработано offset 1600, собрано 700 постов (последний: 2025-11-05)
Обработано offset 1700, собрано 800 постов (последний: 2025-10-26)
Обработано offset 1800, собрано 900 постов (последний: 2025-10-15)
Обработано offset 1900, собрано 1000 постов (последний: 2025-10-05)
Обработано offset 2000, собрано 1100 постов (последний: 2025-09-23)
Обработано offset 2100, собрано 1200 постов (последний: 2025-09-14)
Обработано offset 2200, собрано 1300 постов (последний: 2025-09-05)
Обработано offset 23

In [ ]:
# https://vk.com/msk1_news
group_id = '-211206852'
name = 'msk1_news.csv'

posts = get_posts(api_token, group_id, start_offset=400)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=400
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 400, собрано 100 постов (последний: 2026-01-19)
Обработано offset 500, собрано 200 постов (последний: 2025-12-27)
Обработано offset 600, собрано 300 постов (последний: 2025-12-13)
Обработано offset 700, собрано 400 постов (последний: 2025-11-28)
Обработано offset 800, собрано 500 постов (последний: 2025-11-12)
Обработано offset 900, собрано 600 постов (последний: 2025-10-31)
Обработано offset 1000, собрано 700 постов (последний: 2025-10-17)
Обработано offset 1100, собрано 800 постов (последний: 2025-10-04)
Обработано offset 1200, собрано 900 постов (последний: 2025-09-21)
Обработано offset 1300, собрано 1000 постов (последний: 2025-09-07)
Обработано offset 1400, собрано 1100 постов (последний: 2025-08-26)
Обработано offset 1500, собрано 1200 постов (последний: 2025-08-13)
Обработано offset 1600, собрано 1300 постов (последний: 2025-07-30)
Обработано offset 1700, соб

In [ ]:
# https://vk.com/moskvichmag
group_id = '-167733911'
name = 'moskvichmag.csv'

posts = get_posts(api_token, group_id, start_offset=1500)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1500
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1500, собрано 100 постов (последний: 2026-01-21)
Обработано offset 1600, собрано 200 постов (последний: 2026-01-15)
Обработано offset 1700, собрано 300 постов (последний: 2026-01-06)
Обработано offset 1800, собрано 400 постов (последний: 2025-12-27)
Обработано offset 1900, собрано 500 постов (последний: 2025-12-22)
Обработано offset 2000, собрано 600 постов (последний: 2025-12-16)
Обработано offset 2100, собрано 700 постов (последний: 2025-12-11)
Обработано offset 2200, собрано 800 постов (последний: 2025-12-05)
Обработано offset 2300, собрано 900 постов (последний: 2025-11-28)
Обработано offset 2400, собрано 1000 постов (последний: 2025-11-24)
Обработано offset 2500, собрано 1100 постов (последний: 2025-11-18)
Обработано offset 2600, собрано 1200 постов (последний: 2025-11-12)
Обработано offset 2700, собрано 1300 постов (последний: 2025-11-06)
Обработано offset 28

In [ ]:
# https://vk.com/moslentaru
group_id = '-90864107'
name = 'moslentaru.csv'

posts = get_posts(api_token, group_id, start_offset=1000)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=1000
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 1000, собрано 100 постов (последний: 2026-01-05)
Обработано offset 1100, собрано 200 постов (последний: 2025-12-26)
Обработано offset 1200, собрано 300 постов (последний: 2025-12-17)
Обработано offset 1300, собрано 400 постов (последний: 2025-12-08)
Обработано offset 1400, собрано 500 постов (последний: 2025-11-28)
Обработано offset 1500, собрано 600 постов (последний: 2025-11-18)
Обработано offset 1600, собрано 700 постов (последний: 2025-11-09)
Обработано offset 1700, собрано 800 постов (последний: 2025-10-30)
Обработано offset 1800, собрано 900 постов (последний: 2025-10-21)
Обработано offset 1900, собрано 1000 постов (последний: 2025-10-11)
Обработано offset 2000, собрано 1100 постов (последний: 2025-10-01)
Обработано offset 2100, собрано 1200 постов (последний: 2025-09-23)
Обработано offset 2200, собрано 1300 постов (последний: 2025-09-13)
Обработано offset 23

In [ ]:
# https://vk.com/gazetametro
group_id = '-207490575'
name = 'gazetametro.csv'

posts = get_posts(api_token, group_id, start_offset=700)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=700
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 700, собрано 100 постов (последний: 2026-02-01)
Обработано offset 800, собрано 200 постов (последний: 2025-12-16)
Обработано offset 900, собрано 300 постов (последний: 2025-11-21)
Обработано offset 1000, собрано 400 постов (последний: 2025-11-12)
Обработано offset 1100, собрано 500 постов (последний: 2025-11-04)
Обработано offset 1200, собрано 600 постов (последний: 2025-10-27)
Обработано offset 1300, собрано 700 постов (последний: 2025-10-19)
Обработано offset 1400, собрано 800 постов (последний: 2025-10-12)
Обработано offset 1500, собрано 900 постов (последний: 2025-10-04)
Обработано offset 1600, собрано 1000 постов (последний: 2025-09-26)
Обработано offset 1700, собрано 1100 постов (последний: 2025-09-19)
Обработано offset 1800, собрано 1200 постов (последний: 2025-09-12)
Обработано offset 1900, собрано 1300 постов (последний: 2025-09-05)
Обработано offset 2000, 

In [ ]:
# https://vk.com/ria
group_id = '-15755094'
name = 'ria.csv'

posts = get_posts(api_token, group_id, start_offset=5800)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=5800
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 5800, собрано 100 постов (последний: 2026-01-04)
Обработано offset 5900, собрано 200 постов (последний: 2026-01-02)
Обработано offset 6000, собрано 300 постов (последний: 2025-12-31)
Обработано offset 6100, собрано 400 постов (последний: 2025-12-28)
Обработано offset 6200, собрано 500 постов (последний: 2025-12-27)
Обработано offset 6300, собрано 600 постов (последний: 2025-12-25)
Обработано offset 6400, собрано 700 постов (последний: 2025-12-24)
Обработано offset 6500, собрано 800 постов (последний: 2025-12-22)
Обработано offset 6600, собрано 900 постов (последний: 2025-12-19)
Обработано offset 6700, собрано 1000 постов (последний: 2025-12-18)
Обработано offset 6800, собрано 1100 постов (последний: 2025-12-17)
Обработано offset 6900, собрано 1200 постов (последний: 2025-12-15)
Обработано offset 7000, собрано 1300 постов (последний: 2025-12-13)
Обработано offset 71

In [ ]:
# https://vk.com/tassagency
group_id = '-26284064'
name = 'tassagency.csv'

posts = get_posts(api_token, group_id, start_offset=7800)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset=7800
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 7800, собрано 100 постов (последний: 2026-01-02)
Обработано offset 7900, собрано 200 постов (последний: 2025-12-31)
Обработано offset 8000, собрано 300 постов (последний: 2025-12-30)
Обработано offset 8100, собрано 400 постов (последний: 2025-12-28)
Обработано offset 8200, собрано 500 постов (последний: 2025-12-27)
Обработано offset 8300, собрано 600 постов (последний: 2025-12-25)
Обработано offset 8400, собрано 700 постов (последний: 2025-12-24)
Обработано offset 8500, собрано 800 постов (последний: 2025-12-23)
Обработано offset 8600, собрано 900 постов (последний: 2025-12-21)
Обработано offset 8700, собрано 1000 постов (последний: 2025-12-19)
Обработано offset 8800, собрано 1100 постов (последний: 2025-12-19)
Обработано offset 8900, собрано 1200 постов (последний: 2025-12-17)
Обработано offset 9000, собрано 1300 постов (последний: 2025-12-16)
Обработано offset 91

## Агрегация данных в единый датафрейм

In [ ]:
import os

# все файлы были сохранены в отдельной папке, откуда осуществляется их выгрузка
folder_path = '/content/drive/MyDrive/мага/Диссер/publics'
dataframes = []

# последовательное чтение каждого файла и их присоединение к списку из Dataframe'ов
for file in os.listdir(folder_path):
  file_path = os.path.join(folder_path, file)
  df = pd.read_csv(file_path)
  df['source_nickname'] = file
  print(f'Dataframe {file} содержит {len(df)} публикаций, присоединяем их к основному Dataframe')
  dataframes.append(df)

# преобразование списка из Dataframe'ов в основной Datarame
if dataframes:
  data = pd.concat(dataframes, ignore_index=True)
  data['date'] = pd.to_datetime(data['date'])
  print('')
  print(f'Все готово, основной Dataframe содержит {len(data)} публикаций с 2020-01-01')
  print('')
  data = data.query("date < '2026-01-01'").reset_index(drop=True)
  print(f'Удалены публикации из 2026, сновной Dataframe содержит {len(data)} публикаций с 2020-01-01 по 2025-12-31')

data.to_csv('/content/drive/MyDrive/мага/Диссер/publics/data.csv', index=False)

Dataframe vedomosti.csv содержит 113404 публикаций, присоединяем их к основному Dataframe
Dataframe m24.csv содержит 53559 публикаций, присоединяем их к основному Dataframe
Dataframe typical_msk.csv содержит 7061 публикаций, присоединяем их к основному Dataframe
Dataframe mskagency.csv содержит 9716 публикаций, присоединяем их к основному Dataframe
Dataframe vmdaily.csv содержит 20346 публикаций, присоединяем их к основному Dataframe
Dataframe msk1_news.csv содержит 26676 публикаций, присоединяем их к основному Dataframe
Dataframe moskvichmag.csv содержит 28935 публикаций, присоединяем их к основному Dataframe
Dataframe moslentaru.csv содержит 25726 публикаций, присоединяем их к основному Dataframe
Dataframe gazetametro.csv содержит 17084 публикаций, присоединяем их к основному Dataframe
Dataframe ria.csv содержит 106609 публикаций, присоединяем их к основному Dataframe
Dataframe wow_mosnow.csv содержит 10203 публикаций, присоединяем их к основному Dataframe
Dataframe onmoscow.csv соде